In [1]:
from query_orders import get_spark_session

In [2]:
spark = get_spark_session("alice", "test1234", "lakehouse-local")

26/08/06 03:15:25 WARN Utils: Your hostname, vincent-k-gpu resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/06 03:15:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/vkieuvongngam/.ivy2/cache
The jars for the packages stored in: /home/vkieuvongngam/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-971cb3e9-9c96-447b-9632-ac1c83550f51;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/vkieuvongngam/exploration/spark-lakehouse/spark/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.10.0 in central
:: resolution report :: resolve 78ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.iceberg#iceberg-aws-bundle;1.10.0 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-971cb3e9-9c96-447b-9632-ac1c83550f51
	confs: [default]
	0 artifacts copied, 2 already retrieved (0kB/5ms)
26/08/06 03:15:25 WARN NativeCodeLoader: Un

In [3]:
df = spark.table("sales.orders")

In [4]:
df2 = spark.table("sales.orders_v2")

In [5]:
df.show()

+--------+----------------+-------+
|order_id|        customer| amount|
+--------+----------------+-------+
|       1|       acme-corp| 1250.0|
|       2|      globex-inc|  430.5|
|       3|         initech|  89.99|
|       4|   umbrella-corp|2100.75|
|       5|stark-industries|  560.2|
|      99|         test-co|  12.34|
+--------+----------------+-------+



In [6]:
df2.show()

+--------+----------------+-------+--------------------+
|order_id|        customer| amount|            order_ts|
+--------+----------------+-------+--------------------+
|      99|         test-co|  12.34| 2026-01-01 00:00:00|
|       1|       acme-corp|2500.00|2026-08-06 00:38:...|
|       2|      globex-inc| 861.00|2026-08-06 00:38:...|
|       3|         initech| 179.98|2026-08-06 00:38:...|
|       4|   umbrella-corp|4201.50|2026-08-06 00:38:...|
|       5|stark-industries|1120.40|2026-08-06 00:38:...|
|      99|         test-co|  24.68|2026-08-06 00:38:...|
+--------+----------------+-------+--------------------+



In [7]:
df3 = df.join(df2, on="amount", how="left_anti")

In [8]:
df2.show()

+--------+----------------+-------+--------------------+
|order_id|        customer| amount|            order_ts|
+--------+----------------+-------+--------------------+
|      99|         test-co|  12.34| 2026-01-01 00:00:00|
|       1|       acme-corp|2500.00|2026-08-06 00:38:...|
|       2|      globex-inc| 861.00|2026-08-06 00:38:...|
|       3|         initech| 179.98|2026-08-06 00:38:...|
|       4|   umbrella-corp|4201.50|2026-08-06 00:38:...|
|       5|stark-industries|1120.40|2026-08-06 00:38:...|
|      99|         test-co|  24.68|2026-08-06 00:38:...|
+--------+----------------+-------+--------------------+



In [9]:
from pyspark.sql import functions as F

In [10]:
df2.groupBy("order_id").agg(F.sum("amount").alias("total_amount")).show()

+--------+------------+
|order_id|total_amount|
+--------+------------+
|       5|     1120.40|
|       1|     2500.00|
|       3|      179.98|
|       2|      861.00|
|       4|     4201.50|
|      99|       37.02|
+--------+------------+



In [11]:
result = spark.sql("""
    SELECT order_id, SUM(amount) AS total_amount
    FROM sales.orders
    GROUP BY order_id
""")

In [12]:
result.show()

+--------+------------+
|order_id|total_amount|
+--------+------------+
|       5|       560.2|
|       1|      1250.0|
|       3|       89.99|
|       2|       430.5|
|       4|     2100.75|
|      99|       12.34|
+--------+------------+



In [13]:
result.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- total_amount: double (nullable = true)

